# 09 — Copy vs View & Copy-on-Write (CoW)

The Copy vs View problem is the #1 source of silent bugs in Pandas code.
Pandas 2.0 introduced Copy-on-Write (CoW) to eliminate this entire class of bugs.
This notebook teaches the mechanics, traps, and the new CoW model.

In [ ]:
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")

## 1. What is a View?

A **view** shares memory with the original. Modifying the view modifies the original.
A **copy** is independent. Modifying a copy does NOT affect the original.

In [ ]:
# NumPy views vs copies — the foundation
arr = np.array([10, 20, 30, 40, 50])

view = arr[1:4]      # slice of ndarray = view
copy = arr[1:4].copy()  # explicit copy

view[0] = 999        # modifies arr!
copy[0] = 888        # does NOT modify arr

print(f"Original after view mod: {arr}")
print(f"copy[0] = {copy[0]}")

## 2. Pandas Views — When They Occur

In [ ]:
np.random.seed(42)
df = pd.DataFrame({
    'A': np.arange(10),
    'B': np.arange(10, 20),
    'C': np.arange(20, 30)
})

# Slicing rows with iloc — MAY return a view (unpredictable!)
row_slice = df.iloc[0:5]
print(f"Is row_slice a view? {row_slice._is_view}")

In [ ]:
# Single-column selection — MAY be a view
col_a = df['A']
print(f"Single col is view: {col_a._is_view}")

# Fancy indexing (list of columns) — ALWAYS a copy
two_cols = df[['A', 'B']]
print(f"Two cols is view:   {two_cols._is_view}")

## 3. The SettingWithCopyWarning — The Classic Trap

In [ ]:
import warnings

df = pd.DataFrame({
    'department': ['Eng', 'Sales', 'HR', 'Eng', 'Sales'],
    'salary':     [90000, 80000, 70000, 110000, 75000]
})

# WRONG: chained indexing for write
# df[df['department'] == 'Eng']['salary'] = 999999
# This SILENTLY may or may not modify df — undefined behavior!

print("CHAINED WRITE — raises SettingWithCopyWarning (or silently fails):")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    df[df['department'] == 'Eng']['salary'] = 999999
    if w:
        print(f"Warning: {w[0].category.__name__}: {str(w[0].message)[:80]}")

print("df after chained write (may be unchanged):")
print(df)

In [ ]:
# CORRECT: single loc operation
df.loc[df['department'] == 'Eng', 'salary'] = 999999
print("df after correct loc write:")
print(df)

## 4. When Pandas Returns View vs Copy — Summary

In [ ]:
df = pd.DataFrame(np.arange(20).reshape(4, 5), columns=list('ABCDE'))

operations = [
    ("df['A']",               df['A'],              "column selection"),
    ("df[['A','B']]",         df[['A','B']],         "multi-col (fancy)"),
    ("df.iloc[0:3]",           df.iloc[0:3],          "row slice"),
    ("df.loc[0:3, 'A':'C']",  df.loc[0:3, 'A':'C'],  "loc slice"),
    ("df.copy()",              df.copy(),             "explicit copy"),
]

for op_str, result, label in operations:
    try:
        is_view = result._is_view
    except Exception:
        is_view = 'N/A'
    shares = np.shares_memory(df.values, result.values if hasattr(result, 'values') else result.to_frame().values)
    print(f"{op_str:<28} is_view={is_view!s:<6}  shares_memory={shares}  ({label})")

## 5. The Safe Pattern — Always Use .copy()

In [ ]:
np.random.seed(42)
employees = pd.DataFrame({
    'department': np.random.choice(['Engineering', 'Sales', 'HR'], 20),
    'salary':     np.random.randint(50000, 120000, 20)
})

# SAFE: always call .copy() when you plan to modify a subset
engineers = employees[employees['department'] == 'Engineering'].copy()
engineers['salary'] = engineers['salary'] * 1.10   # 10% raise
engineers['bonus'] = engineers['salary'] * 0.10

print("Engineers with raise (copy):")
print(engineers.head())
print(f"\nOriginal salary unchanged: {employees.loc[employees['department'] == 'Engineering', 'salary'].mean():.0f}")

## 6. Copy-on-Write (CoW) — Pandas 2.0+

**Copy-on-Write** makes all indexing operations return a **logical copy**.
The actual copy is deferred until a write happens — so read operations have zero overhead.
This eliminates SettingWithCopyWarning entirely.

In [ ]:
# Enable CoW (Pandas 2.0+)
# In Pandas 3.0, CoW will be the only mode
pd.options.mode.copy_on_write = True
print("Copy-on-Write enabled")

In [ ]:
df = pd.DataFrame({
    'A': [1, 2, 3, 4, 5],
    'B': [10, 20, 30, 40, 50]
})

# With CoW: subset is always independent on write — NO silent mutations
subset = df.iloc[0:3]  # looks like a view
subset['A'] = 999       # with CoW: modifies subset only, NOT df

print("subset:")
print(subset)
print("\ndf (unchanged with CoW):")
print(df)

In [ ]:
# CoW: chained write also works correctly (no silent failure)
df2 = pd.DataFrame({'dept': ['Eng', 'Sales', 'Eng'], 'salary': [90000, 80000, 110000]})

# This is safe with CoW — no SettingWithCopyWarning
df2[df2['dept'] == 'Eng']['salary'] = 999999

# But it STILL doesn't modify df2 — you need loc for that
print("df2 (chained write still doesn't modify original with CoW):")
print(df2)
print("\nAlways use loc for writes, CoW or not.")

In [ ]:
# CoW performance: lazy copy — copy happens only at write time
import time

big = pd.DataFrame({'A': np.arange(1_000_000), 'B': np.arange(1_000_000)})

# Read-only access — no copy made
start = time.perf_counter()
view = big.iloc[0:500_000]   # no actual copy made
total = view['A'].sum()       # read only — fast
t_read = time.perf_counter() - start

# Write access — copy happens now
start = time.perf_counter()
view2 = big.iloc[0:500_000]
view2['A'] = 0               # triggers copy of 500k rows
t_write = time.perf_counter() - start

print(f"Read (no copy):        {t_read*1000:.1f}ms")
print(f"Write (triggers copy): {t_write*1000:.1f}ms")

## 7. Diagnosing View vs Copy with np.shares_memory()

In [ ]:
# Disable CoW for this demo to show classic behavior
pd.options.mode.copy_on_write = False

df = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]})

single_col = df['A']
print(f"df['A'] shares memory: {np.shares_memory(df['A'].values, single_col.values)}")

fancy = df[['A']]
print(f"df[['A']] shares memory: {np.shares_memory(df['A'].values, fancy['A'].values)}")

explicit_copy = df.copy()
print(f"df.copy() shares memory: {np.shares_memory(df['A'].values, explicit_copy['A'].values)}")

## 8. Method Chaining and CoW

In [ ]:
pd.options.mode.copy_on_write = True

np.random.seed(42)
employees = pd.DataFrame({
    'name':       [f'Emp_{i}' for i in range(20)],
    'department': np.random.choice(['Engineering', 'Sales', 'HR'], 20),
    'salary':     np.random.randint(50000, 150000, 20)
})

# With CoW: method chaining is completely safe
# Each step returns a new logical copy — writes are deferred
result = (
    employees
    .query("salary > 80000")
    .assign(bonus=lambda df: df['salary'] * 0.10)
    .sort_values('salary', ascending=False)
    .reset_index(drop=True)
)
print(result)

## 9. Golden Rules — View, Copy, CoW

**Pre-CoW (Pandas < 2.0):**
1. Always call `.copy()` when creating a subset you plan to modify
2. Never use chained indexing for writes — always use single `.loc[mask, col] = val`
3. Use `np.shares_memory()` to diagnose copy vs view issues

**With CoW (Pandas 2.0+):**
1. Enable with `pd.options.mode.copy_on_write = True`
2. All indexing results are logically independent — writes are deferred
3. Still use `.loc` for writes — clearer intent, same behavior
4. In Pandas 3.0, CoW is the default and only mode

**Universal rule**: If you want to modify a subset, use `loc` or create an explicit `.copy()`.